In [129]:
import pandas
import matplotlib.pyplot as plt
import numpy
import math
import os
from typing import List
from dataclasses import dataclass

@dataclass
class stefan_variables:
    c_p: float = 1
    kappa: float = 1
    rho: float = 1
    T_l: float = 1
    T_m: float = 1
    h_m: float = 1
    alpha: float = kappa/(rho*c_p)


def one_GaussLegendreQuadrature(self, n):
    N = n - 1
    N1 = n
    N2 = n + 1
    xu = numpy.linspace(-1, 1, N1).reshape(-1,1)
    x= - numpy.cos((2*(numpy.linspace(0,N,N1).reshape(-1,1)) + 1)*numpy.pi/(2*N+2)) + (0.27/N1)*numpy.sin(numpy.pi*xu*N/N2)
    L = numpy.zeros((n, n+1))
    x0 = 2

    while numpy.max(numpy.absolute(x-x0)) > numpy.finfo(numpy.float64).eps:
        L[:, 0] = 1
        L[: , 1] = numpy.transpose(x)
        for i in range(1,n):
            temp_Li = ((2*i+1)*(x.T*L[:,i]) - i*L[:,i-1])/(i+1)
            L[:, i+1] = numpy.reshape(temp_Li, (1,-1))
        temp_dLp = ((n)*(x.T*L[:, n]) - (n)*L[:, n-1])/((x.T*x.T) - 1)
        dLp = temp_dLp
        x0 = x
        x = x0 - numpy.reshape((L[:, n]/dLp),(-1,1))
    w = (2/((1-(x.T*x.T))*(dLp*dLp)))
    return x.T, w


class legendreGaussQuad():
    def __init__(self, ngp, dim):
        self.gps1d, self.w1d = self.oneD_GaussLegendreQuadrature(ngp)
        self.gps, self.w, self.ngp = self.multiD_GaussLegendreQuadrature(ngp, dim)
    
    def oneD_GaussLegendreQuadrature(self, n):
        N = n - 1
        N1 = n
        N2 = n + 1
        xu = numpy.linspace(-1, 1, N1).reshape(-1,1)
        x= - numpy.cos((2*(numpy.linspace(0,N,N1).reshape(-1,1)) + 1)*numpy.pi/(2*N+2)) + (0.27/N1)*numpy.sin(numpy.pi*xu*N/N2)
        L = numpy.zeros((n, n+1))
        x0 = 2

        while numpy.max(numpy.absolute(x-x0)) > numpy.finfo(numpy.float64).eps:
            L[:, 0] = 1
            L[: , 1] = numpy.transpose(x)
            for i in range(1,n):
                #using the recurrence relation:
                # (n+1)*P_n+1 = (2n+1)*x*P_n - n*P_n-1
                temp_Li = ((2*i+1)*(x.T*L[:,i]) - i*L[:,i-1])/(i+1)
                L[:, i+1] = numpy.reshape(temp_Li, (1,-1))
            #using the recurrence relation:
            #(1-x^2)*P'_n = (n)*[P_n-1 - x*P_n]
            temp_dLp = ((n)*(x.T*L[:, n]) - (n)*L[:, n-1])/((x.T*x.T) - 1)
            dLp = temp_dLp
            x0 = x
            x = x0 - numpy.reshape((L[:, n]/dLp),(-1,1))
        w = (2/((1-(x.T*x.T))*(dLp*dLp)))
        return x.T, w
    
    def multiD_GaussLegendreQuadrature(self, ngp, dim):
        gps = numpy.zeros((ngp**dim, dim))
        gps[:, 0] = numpy.tile(self.gps1d, ngp**(dim-1))
        if dim == 1:
            w = numpy.reshape(self.w1d, (-1,1))
        elif dim == 3:
            gps[:, 1] = numpy.tile(numpy.repeat(self.gps1d,ngp),ngp)
            gps[:, 2] = numpy.repeat(self.gps1d, ngp**(dim-1))
        else:
            gps[:, 1] = numpy.repeat(self.gps1d, ngp**(dim-1))
            w1 = numpy.tile(self.w1d, ngp**(dim-1))
            w2 = numpy.repeat(self.w1d, ngp**(dim-1))
            w = (w1*w2).T
        
        return gps, w, ngp**dim
    
## quadrature scaling: w_scaled_j = ((b-a)/(d-c))/w_j -> for [c,d] = [-1,1], w_scaled_j = (b-a)/2 * w_j
##                     x_scaled_j = c + (d-c)/(b-a) * (x_j-a) -> for [c,d] = [-1, 1], s_scaled_j = -1 + 2/(b-a) * (x_j-a) 
    def integrate_func(self, func, lim):
        lim_b = lim[1][-1]
        a = float(lim[0])
        b = float(lim_b)
        integral = 0
        #print(self.w1d)
        #print(self.gps1d)
        for wx, gpsx in zip(self.w, self.gps):
            #print(gpsx)
            integral += wx*func((b-a)*0.5*gpsx + (b+a)*0.5)
            #print(integral)
        #print('-----', (b-a)*0.5*integral)
        return (b-a)*0.5*integral



In [130]:
quadr = legendreGaussQuad(10,1)

In [131]:
quadr.gps

array([[-0.97390653],
       [-0.86506337],
       [-0.67940957],
       [-0.43339539],
       [-0.14887434],
       [ 0.14887434],
       [ 0.43339539],
       [ 0.67940957],
       [ 0.86506337],
       [ 0.97390653]])

In [114]:
#T = T_l - (T_l - T_m)*erf(x/2*sqrt(alpha*t))/erf(X(t)/2*sqrt(alpha*t)) for 0 <= x < X(t), 0 < t < t_end
#X = 2*sqrt(alpha)*lambda*sqrt(t), for 0 < t < t_end
#alpha = kappa/(rho*c_p) 
#lambda is the unique root of the monotonic function,
#f(lambda) = c_p*((T_m - T_l)/h_m) * exp(-1*lambda**2)/(sqrt(pi)*erf(lambda)) - lambda

#erf(x) := (2/pi) * int^x_0 exp(-1*y**2) dy


from scipy.integrate import quad
from scipy.optimize import fsolve
from scipy import special

def sci_erf(x: float = None):
    return special.erf(x)

def errfunc(x: float = None):
    erf = numpy.exp(-x**2)
    return erf

def integrate_erf(x: float = None, quadr: legendreGaussQuad = None):
    integral = 2/(numpy.sqrt(math.pi))*quadr.integrate_func(errfunc, [0, x])
   #integral = sci_erf(x)
    return integral 

def return_lambda(c_p: float= None, T_m: float= None, T_l: float= None, h_m: float = None,  quadr: legendreGaussQuad = None, lam: float = None):
    def func(x):
        erf = integrate_erf(x, quadr)
        print(x, erf)
        return c_p*((T_m - T_l)/h_m) * numpy.exp(-1*(x**2))/(numpy.sqrt(math.pi)*erf) - x
    
    roots = fsolve(func, 6)
    return roots 

def interface_position(variables: stefan_variables, t_all: List[float] = None, t: float = None):
    c_p = variables['c_p']
    kappa = variables['kappa']
    rho = variables['rho']
    T_l = variables['T_l']
    T_m = variables['T_m']
    h_m = variables['h_m']
    alpha = variables['alpha']
    if t is None and t_all is not None:
        X = []
        for idx, t_i in enumerate(t_all):
            X[idx] = 2*numpy.sqrt(alpha*t_i)*lam
        return X
    elif t_all is None and t is not None:
        X = 2*numpy.sqrt(alpha*t)*lam

def stefan_temperature():

In [115]:
stefan_variables(
    c_p = 4200,
    kappa = 0.6,
    rho = 1000,
    T_l = 300,
    T_m = 273,
    h_m = 333700
)

val = return_lambda(c_p= c_p, T_m= T_m, T_l= T_l, h_m= h_m, quadr= quadr, lam= 4)

[6] [1.00001122]
[6.] [1.00001122]
[6.] [1.00001122]
[6.00000009] [1.00001122]
[-1.30864208e-11] [-1.47664446e-11]
[6.] [1.00001122]


In [ ]:
X = 

In [118]:
alpha

1.4285714285714285e-07